<a href="https://colab.research.google.com/github/Aqifcodes/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Aqifcodes/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

### Rule Definition & Reason Codes

**Rule:**  
Target content items ranking between positions 4 and 20 (Page 1/2) that collect high search impressions but yield below-average Click-Through Rates (CTR).

$$\text{Baseline Score} = \left( \frac{\text{total\_impressions}}{\text{avg\_position} + 1} \right) \times (1 - \text{CTR})$$

**Reason Codes & Actions:**
* `HIGH_IMPRESSION_LOW_CTR`: Content is visible in search results (positions 4–20) but fails to convert impressions into clicks.
* **Action Label:** `REWRITE_TITLE_AND_META`

In [4]:
import duckdb
import os
import pandas as pd
import numpy as np
from google.colab import userdata

# Authenticate DuckDB with Hugging Face token
hf_token = userdata.get('HF_TOKEN')
con = duckdb.connect()
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{hf_token}')")

data_path = "hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet"

# Verify base metrics on mid-panel month
summary = con.execute(f"""
    SELECT
        COUNT(DISTINCT content_hash_id) AS total_items,
        SUM(gsc_impressions) AS total_impressions,
        SUM(gsc_clicks) AS total_clicks,
        AVG(gsc_avg_position) AS avg_position
    FROM read_parquet('{data_path}')
""").df()

print("=== DATASET SUMMARY (month=2026-03) ===")
print(summary)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

=== DATASET SUMMARY (month=2026-03) ===
   total_items  total_impressions  total_clicks  avg_position
0       331437        280657589.0      821832.0     15.826651


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [6]:
# 1. Aggregate daily performance into content-level metrics
df_aggregated = con.execute(f"""
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_clicks) AS total_clicks,
        SUM(gsc_impressions) AS total_impressions,
        AVG(gsc_avg_position) AS avg_position,
        SUM(ga4_sessions) AS total_sessions,
        CASE
            WHEN SUM(gsc_impressions) > 0 THEN SUM(gsc_clicks) * 1.0 / SUM(gsc_impressions)
            ELSE 0
        END AS ctr
    FROM read_parquet('{data_path}')
    GROUP BY client_hash_id, content_hash_id
""").df()

# 2. Compute Baseline Priority Score
df_aggregated['baseline_score'] = np.where(
    (df_aggregated['avg_position'] >= 4) & (df_aggregated['avg_position'] <= 20),
    (df_aggregated['total_impressions'] / (df_aggregated['avg_position'] + 1)) * (1 - df_aggregated['ctr']),
    0
)

# 3. Assign Reason Code and Action Label
df_aggregated['reason_code'] = 'HIGH_IMPRESSION_LOW_CTR'
df_aggregated['action_label'] = 'REWRITE_TITLE_AND_META'

# 4. Rank descending by baseline score
ranked_queue = df_aggregated.sort_values(by='baseline_score', ascending=False).reset_index(drop=True)

# 5. Format output DataFrame
output_df = ranked_queue[[
    'client_hash_id',
    'content_hash_id',
    'baseline_score',
    'reason_code',
    'action_label',
    'total_impressions',
    'avg_position',
    'ctr'
]]

# 6. Export CSV to work/outputs/ directory
os.makedirs('work/outputs', exist_ok=True)
os.makedirs('../outputs', exist_ok=True)

output_path = 'work/outputs/baseline_action_score.csv'
output_df.to_csv(output_path, index=False)
output_df.to_csv('../outputs/baseline_action_score.csv', index=False)

print(f"Successfully written ranked queue to {output_path}")
print(f"Total scored rows in queue: {len(output_df)}")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Successfully written ranked queue to work/outputs/baseline_action_score.csv
Total scored rows in queue: 331437


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

### Top-20 Action Review & Validation

* **Action:** `REWRITE_TITLE_AND_META`
* **Reason Code:** `HIGH_IMPRESSION_LOW_CTR`
* **Confidence Note:** High confidence for non-branded informational queries ranking on positions 4–10. Moderate confidence for positions 11–20.

**Top-20 Failure Modes ("What would make it wrong"):**
1. **Zero-Click Search Intent:** Google displays a featured snippet or direct answer box, satisfying search intent on the SERP without requiring a click.
2. **Brand / Navigation Queries:** The page accidentally ranks for broad competitor brand queries where users intentionally click the competitor site.
3. **Intent Mismatch:** The snippet ranks for a high-volume broader topic, but the page content covers a niche sub-topic.
4. **Out-of-Stock / Seasonal Content:** The content represents temporary campaigns or out-of-stock product pages.

In [7]:
# Display top 20 ranked items
top_20 = output_df.head(20)
print("=== TOP 20 RANKED RECOMMENDATIONS ===")
print(top_20[['content_hash_id', 'baseline_score', 'total_impressions', 'avg_position', 'ctr']])

=== TOP 20 RANKED RECOMMENDATIONS ===
             content_hash_id  baseline_score  total_impressions  avg_position  \
0   content_e7b5dd4dff461ad2    36542.494347           205045.0      4.544203   
1   content_b99ea6861864dea5    35591.232358           194337.0      4.450106   
2   content_471d9cabce329a66    29082.057132           164885.0      4.656030   
3   content_44f34c0a90047651    25444.149419           212404.0      7.346909   
4   content_8e1334d6356668e3    24340.638042           134984.0      4.545582   
5   content_f43118e089ecc69a    23064.186128           139417.0      5.036458   
6   content_e578ac84778da489    22950.360442           117764.0      4.124146   
7   content_95ff62babbfac9c7    22177.732806           122857.0      4.529104   
8   content_b17c1d1cb0a346d6    20395.322473           115947.0      4.661347   
9   content_00d4fdf6e48a2d38    20039.786638           126836.0      5.305905   
10  content_7c6373141eae744a    19518.285168           132593.0      5.

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

### Weak Picks & Leakage Audit

**Weak Pick Identification:**
* Items with high `avg_position` (> 15) and massive impression volume often turn out to be accidental rankings for irrelevantly broad search terms. Rewriting title tags on these will not improve CTR.

**Leakage Prevention Check:**
* **No Future Windows:** All aggregations use strictly past daily metrics within `month=2026-03`.
* **No Label/Product Leakage:** Scores depend solely on raw observed GSC performance metrics (`gsc_impressions`, `gsc_position`, `gsc_clicks`). No downstream outcome flags or future month data (`2026-04` or later) were accessed.

In [8]:
# Verification query: Confirm no missing values or invalid score artifacts
print("=== WEAK PICK & DATA INTEGRITY AUDIT ===")
print(f"Total rows with baseline_score > 0: {(output_df['baseline_score'] > 0).sum()}")
print(f"Max Baseline Score: {output_df['baseline_score'].max():.2f}")
print(f"Min Baseline Score: {output_df['baseline_score'].min():.2f}")
print(f"Null values in score: {output_df['baseline_score'].isnull().sum()}")

=== WEAK PICK & DATA INTEGRITY AUDIT ===
Total rows with baseline_score > 0: 103115
Max Baseline Score: 36542.49
Min Baseline Score: 0.00
Null values in score: 0


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.